# 面试问题：ORPO 怎样在不加载 Reference Model 的情况下同时做 SFT 与偏好优化？

        ## 可直接复述的回答主线

        1. ORPO 在 chosen 的监督负对数似然之外，加入 chosen/rejected 对数优势比的偏好损失。
2. 它只需要当前策略模型，不需要像 DPO 那样额外执行冻结 Reference Model，因此减少训练显存与前向开销。
3. 核心中间量是 chosen log-prob、rejected log-prob、log-odds margin、pair accuracy 和梯度范数。
4. Baseline 应使用相同初始化和同一批偏好对，只训练 chosen SFT，才能公平比较 margin。
5. 序列概率接近 1 时直接计算 log(1-p) 会出现负无穷，必须对概率上界做数值保护。
6. 生产训练仍需真实自回归逐 Token mask、长度归一化策略、混合精度、分布式训练、验证集与安全评测。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是六条中文客服偏好对，每条都有可读 prompt、chosen 和 rejected。为了离线快速执行，模型是手写的条件 unigram 小网络：对 prompt embedding 做均值池化后预测回复词分布；它能真实 forward/backward，却不代表完整自回归 LLM。

In [1]:
import math  # 计算手写梯度范数。
import torch  # 使用基础 PyTorch 张量、Module 和自动微分执行真实训练。
torch.manual_seed(57)  # 固定模型初始化和训练轨迹。
preference_pairs = [{"id": "orpo-01", "prompt": "退款 审核 通过", "chosen": "三个 工作日 原路 到账", "rejected": "一定 立刻 到账"}, {"id": "orpo-02", "prompt": "用户 要求 重置 密码", "chosen": "先 完成 身份 验证 再 重置", "rejected": "直接 告诉 我 验证码"}, {"id": "orpo-03", "prompt": "包裹 四十八 小时 未更新", "chosen": "提交 催件 并 保留 单号", "rejected": "不用 处理 继续 等"}, {"id": "orpo-04", "prompt": "客户 询问 续费", "chosen": "设置 页面 关闭 下周期 生效", "rejected": "无法 关闭 永久 扣款"}, {"id": "orpo-05", "prompt": "发票 抬头 修改", "chosen": "开票 前 修改 公司 名称", "rejected": "开票 后 随意 覆盖"}, {"id": "orpo-06", "prompt": "账户 异常 登录", "chosen": "冻结 会话 并 完成 实名 验证", "rejected": "共享 密码 给 客服"}]  # 定义六条具有安全和业务语义的偏好训练样本。
all_text = " ".join(record["prompt"] + " " + record["chosen"] + " " + record["rejected"] for record in preference_pairs)  # 合并全部训练文本以构造教学词表。
vocabulary = ["<unk>"] + sorted(set(all_text.split()))  # 建立包含未知词的确定性空格词表。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立词到整数编号的映射。
def encode(text):  # 把空格分词文本转换为 PyTorch 长整数张量。
    return torch.tensor([token_to_id.get(token, 0) for token in text.split()], dtype=torch.long)  # 返回当前句子的 Token ID 序列。
encoded_pairs = [{**record, "prompt_ids": encode(record["prompt"]), "chosen_ids": encode(record["chosen"]), "rejected_ids": encode(record["rejected"])} for record in preference_pairs]  # 预编码六条偏好对。
print("教学实验输入：六条客服偏好对")  # 标记下方为小型脱敏训练集。
print("样本      prompt                 chosen                              rejected")  # 输出偏好数据预览表头。
for record in preference_pairs:  # 逐样本展示自然语言而不是裸 Token。
    print(f"{record['id']:<9} {record['prompt']:<22} {record['chosen']:<35} {record['rejected']}")  # 输出当前 prompt 和偏好回复。
print("词表大小=", len(vocabulary), "首条Token=", encoded_pairs[0]["prompt_ids"].tolist(), encoded_pairs[0]["chosen_ids"].tolist())  # 展示张量化输入和词表规模。

教学实验输入：六条客服偏好对
样本      prompt                 chosen                              rejected
orpo-01   退款 审核 通过               三个 工作日 原路 到账                        一定 立刻 到账
orpo-02   用户 要求 重置 密码            先 完成 身份 验证 再 重置                     直接 告诉 我 验证码
orpo-03   包裹 四十八 小时 未更新          提交 催件 并 保留 单号                       不用 处理 继续 等
orpo-04   客户 询问 续费               设置 页面 关闭 下周期 生效                     无法 关闭 永久 扣款
orpo-05   发票 抬头 修改               开票 前 修改 公司 名称                       开票 后 随意 覆盖
orpo-06   账户 异常 登录               冻结 会话 并 完成 实名 验证                    共享 密码 给 客服
词表大小= 66 首条Token= [59, 28, 60] [2, 33, 19, 15]


## 2. Baseline / 基线：相同初始化，只对 chosen 做 SFT

基线最小化 chosen 回复的平均负 log-prob，不直接看 rejected。训练循环会真实调用 `forward()`、`backward()` 和手动参数更新。

In [2]:
class TinyConditionalLM(torch.nn.Module):  # 定义具有显式 forward 的条件 unigram 教学模型。
    def __init__(self, vocab_size, hidden_dim=18):  # 初始化 prompt embedding 和词表投影层。
        super().__init__()  # 注册 PyTorch 模块参数。
        self.embedding = torch.nn.Embedding(vocab_size, hidden_dim)  # 将 prompt Token 映射到隐藏空间。
        self.output = torch.nn.Linear(hidden_dim, vocab_size)  # 根据 prompt 上下文预测回复词分布。
    def forward(self, prompt_ids):  # 对一条变长 prompt 执行条件前向。
        embedded = self.embedding(prompt_ids)  # 取得每个 prompt Token 的隐藏向量。
        context = torch.tanh(embedded.mean(dim=0))  # 均值池化并加入非线性得到条件上下文。
        logits = self.output(context)  # 输出整个回复词表的未归一化分数。
        return logits  # 返回供序列 log-prob 使用的词表 logits。
def response_log_probability(model, prompt_ids, response_ids):  # 计算教学模型对一条回复的平均条件 log-prob。
    logits = model(prompt_ids)  # 调用模型显式 forward 获取词表 logits。
    log_probs = torch.log_softmax(logits, dim=-1)  # 手动归一化为对数概率。
    selected = log_probs[response_ids]  # 读取回复中每个词的条件 log-prob。
    return selected.mean()  # 用平均值降低不同回复长度的直接偏置。
initial_model = TinyConditionalLM(len(vocabulary))  # 创建所有实验共享的确定性初始化模型。
initial_state = {name: value.detach().clone() for name, value in initial_model.state_dict().items()}  # 深拷贝初始化以确保 SFT 和 ORPO 公平起点。
def train_candidate(mode, steps=100, learning_rate=0.12, beta=1.2):  # 手写 SFT 或 ORPO 的全批次训练循环。
    model = TinyConditionalLM(len(vocabulary))  # 创建当前候选模型。
    model.load_state_dict(initial_state)  # 恢复与另一候选完全相同的初始化。
    history = []  # 保存损失、margin、准确率和梯度范数轨迹。
    for step in range(steps):  # 执行固定次数的真实梯度更新。
        model.zero_grad(set_to_none=True)  # 清除上一步累积梯度。
        chosen_logps = torch.stack([response_log_probability(model, record["prompt_ids"], record["chosen_ids"]) for record in encoded_pairs])  # 前向计算六条 chosen 平均 log-prob。
        rejected_logps = torch.stack([response_log_probability(model, record["prompt_ids"], record["rejected_ids"]) for record in encoded_pairs])  # 前向计算六条 rejected 平均 log-prob。
        sft_loss = -chosen_logps.mean()  # 计算 chosen 监督负对数似然。
        chosen_probability = torch.exp(chosen_logps).clamp(max=1.0 - 1.0e-6)  # 把 chosen 概率限制在开区间内保证 log(1-p) 有限。
        rejected_probability = torch.exp(rejected_logps).clamp(max=1.0 - 1.0e-6)  # 对 rejected 概率应用同样数值保护。
        chosen_log_odds = chosen_logps - torch.log1p(-chosen_probability)  # 计算 chosen 的稳定 log-odds。
        rejected_log_odds = rejected_logps - torch.log1p(-rejected_probability)  # 计算 rejected 的稳定 log-odds。
        log_odds_margin = chosen_log_odds - rejected_log_odds  # 计算每条偏好对的优势比差。
        preference_loss = -torch.nn.functional.logsigmoid(log_odds_margin).mean()  # 用 logistic 损失鼓励 chosen odds 高于 rejected。
        loss = sft_loss if mode == "sft" else sft_loss + beta * preference_loss  # 按候选模式组合单模型 ORPO 目标。
        loss.backward()  # 对真实模型参数执行反向传播。
        gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters()))  # 汇总本步全部参数的二范数梯度。
        with torch.no_grad():  # 在无梯度环境中手动执行 SGD。
            for parameter in model.parameters():  # 遍历 embedding、投影权重和偏置。
                parameter.add_(parameter.grad, alpha=-learning_rate)  # 使用固定学习率更新当前参数。
        if step % 20 == 0 or step == steps - 1:  # 每二十步记录一行可读训练轨迹。
            pair_accuracy = (chosen_logps > rejected_logps).to(torch.float64).mean().item()  # 统计 chosen 概率高于 rejected 的比例。
            history.append({"step": step, "loss": loss.item(), "sft": sft_loss.item(), "preference": preference_loss.item(), "margin": (chosen_logps - rejected_logps).mean().item(), "pair_accuracy": pair_accuracy, "gradient_norm": gradient_norm})  # 保存当前损失和偏好中间量。
    return model, history  # 返回训练后模型和完整检查点轨迹。
sft_model, sft_history = train_candidate("sft")  # 实际训练 chosen-only 基线模型。
print("SFT Baseline 训练轨迹")  # 标记下表展示真实 forward/backward 结果。
print("step    loss      chosen_NLL  pref_loss  margin   pair_acc  grad_norm")  # 输出基线训练轨迹表头。
for row in sft_history:  # 逐检查点展示 SFT 变化。
    print(f"{row['step']:>4} {row['loss']:>9.5f} {row['sft']:>11.5f} {row['preference']:>10.5f} {row['margin']:>8.4f} {row['pair_accuracy']:>9.1%} {row['gradient_norm']:>10.5f}")  # 输出当前基线损失、margin 和梯度。

SFT Baseline 训练轨迹
step    loss      chosen_NLL  pref_loss  margin   pair_acc  grad_norm
   0   4.24110     4.24110    0.64484   0.1060     66.7%    0.36557
  20   3.93133     3.93133    0.51196   0.4028    100.0%    0.35254
  40   3.64407     3.64407    0.40391   0.6899    100.0%    0.33883
  60   3.38013     3.38013    0.31743   0.9673    100.0%    0.32371
  80   3.14099     3.14099    0.24926   1.2340    100.0%    0.30681
  99   2.93800     2.93800    0.19858   1.4761    100.0%    0.28916


## 3. ORPO 核心实现与中间过程

ORPO 复用同一个 `train_candidate`，但损失为 `chosen_NLL + beta * -logσ(log_odds_chosen-log_odds_rejected)`。没有 Reference Model，也没有隐藏第二次模型前向。

In [3]:
orpo_model, orpo_history = train_candidate("orpo")  # 从同一初始化执行 ORPO 真实训练。
print("ORPO 训练轨迹")  # 标记下表展示偏好损失如何推动 margin。
print("step    total_loss  chosen_NLL  pref_loss  margin   pair_acc  grad_norm")  # 输出 ORPO 轨迹表头。
for row in orpo_history:  # 逐检查点展示两项损失和梯度。
    print(f"{row['step']:>4} {row['loss']:>10.5f} {row['sft']:>11.5f} {row['preference']:>10.5f} {row['margin']:>8.4f} {row['pair_accuracy']:>9.1%} {row['gradient_norm']:>10.5f}")  # 输出当前 ORPO 中间量。
first_record = encoded_pairs[0]  # 选择退款样本展示最终 odds 分解。
first_chosen_logp = response_log_probability(orpo_model, first_record["prompt_ids"], first_record["chosen_ids"]).detach()  # 计算退款 chosen 最终 log-prob。
first_rejected_logp = response_log_probability(orpo_model, first_record["prompt_ids"], first_record["rejected_ids"]).detach()  # 计算退款 rejected 最终 log-prob。
first_chosen_odds = first_chosen_logp - torch.log1p(-torch.exp(first_chosen_logp).clamp(max=1.0 - 1.0e-6))  # 计算 chosen 稳定 log-odds。
first_rejected_odds = first_rejected_logp - torch.log1p(-torch.exp(first_rejected_logp).clamp(max=1.0 - 1.0e-6))  # 计算 rejected 稳定 log-odds。
print(f"orpo-01中间量：chosen_logp={first_chosen_logp.item():.5f}，rejected_logp={first_rejected_logp.item():.5f}，log_odds_margin={(first_chosen_odds-first_rejected_odds).item():.5f}")  # 展示 ORPO 公式各项真实数值。

ORPO 训练轨迹
step    total_loss  chosen_NLL  pref_loss  margin   pair_acc  grad_norm
   0    5.01491     4.24110    0.64484   0.1060     66.7%    0.61662
  20    4.25794     3.79439    0.38629   0.7439    100.0%    0.50880
  40    3.72606     3.43119    0.24572   1.2572    100.0%    0.43418
  60    3.32939     3.13162    0.16481   1.6853    100.0%    0.37931
  80    3.02268     2.88398    0.11558   2.0514    100.0%    0.33553
  99    2.79188     2.68927    0.08551   2.3541    100.0%    0.30041
orpo-01中间量：chosen_logp=-2.55121，rejected_logp=-4.80805，log_odds_margin=2.32984


## 4. 逐样本结果与结果解读

对同一六条偏好对计算 SFT 与 ORPO 的 chosen/rejected log-prob 和 margin。这里的目标是比较机制，不声称小模型结果能泛化到真实 LLM。

In [4]:
def evaluate_model(model):  # 逐偏好对评估 chosen 与 rejected 概率关系。
    rows = []  # 保存六条样本的 log-prob 和 margin。
    with torch.no_grad():  # 评估阶段不构建梯度图。
        for record in encoded_pairs:  # 逐条运行同一模型 forward。
            chosen_logp = response_log_probability(model, record["prompt_ids"], record["chosen_ids"]).item()  # 计算当前 chosen 平均 log-prob。
            rejected_logp = response_log_probability(model, record["prompt_ids"], record["rejected_ids"]).item()  # 计算当前 rejected 平均 log-prob。
            rows.append({"id": record["id"], "chosen": chosen_logp, "rejected": rejected_logp, "margin": chosen_logp - rejected_logp, "correct": chosen_logp > rejected_logp})  # 保存当前偏好结果。
    return rows  # 返回逐样本评估表。
sft_rows = evaluate_model(sft_model)  # 评估 chosen-only SFT 候选。
orpo_rows = evaluate_model(orpo_model)  # 评估单模型 ORPO 候选。
sft_mean_margin = sum(row["margin"] for row in sft_rows) / len(sft_rows)  # 计算 SFT 平均偏好 margin。
orpo_mean_margin = sum(row["margin"] for row in orpo_rows) / len(orpo_rows)  # 计算 ORPO 平均偏好 margin。
sft_pair_accuracy = sum(row["correct"] for row in sft_rows) / len(sft_rows)  # 计算 SFT pair accuracy。
orpo_pair_accuracy = sum(row["correct"] for row in orpo_rows) / len(orpo_rows)  # 计算 ORPO pair accuracy。
print("样本      SFT chosen/rejected/margin       ORPO chosen/rejected/margin      ORPO正确")  # 输出逐样本同数据对照表头。
for sft_row, orpo_row in zip(sft_rows, orpo_rows):  # 逐偏好对比较训练目标效果。
    print(f"{sft_row['id']:<9} {sft_row['chosen']:>7.3f}/{sft_row['rejected']:>7.3f}/{sft_row['margin']:>7.3f}       {orpo_row['chosen']:>7.3f}/{orpo_row['rejected']:>7.3f}/{orpo_row['margin']:>7.3f}       {str(orpo_row['correct']):>7}")  # 输出当前样本的两种 margin。
print(f"结果解读：SFT平均margin={sft_mean_margin:.4f}、pair accuracy={sft_pair_accuracy:.1%}；ORPO平均margin={orpo_mean_margin:.4f}、pair accuracy={orpo_pair_accuracy:.1%}。")  # 解释显式偏好项带来的变化。

样本      SFT chosen/rejected/margin       ORPO chosen/rejected/margin      ORPO正确
orpo-01    -2.774/ -4.229/  1.455        -2.551/ -4.808/  2.257          True
orpo-02    -3.100/ -4.573/  1.474        -2.826/ -5.286/  2.460          True
orpo-03    -3.071/ -4.439/  1.367        -2.760/ -5.068/  2.308          True
orpo-04    -2.840/ -4.495/  1.656        -2.649/ -5.069/  2.420          True
orpo-05    -2.640/ -4.172/  1.531        -2.415/ -4.860/  2.445          True
orpo-06    -3.143/ -4.590/  1.448        -2.879/ -5.203/  2.324          True
结果解读：SFT平均margin=1.4885、pair accuracy=100.0%；ORPO平均margin=2.3690、pair accuracy=100.0%。


## 5. 失败案例与修正：概率等于 1 时 log(1-p) 溢出

直接对 `logp=0` 计算 `log(1-exp(logp))` 会得到负无穷。训练实现先把概率裁到 `1-1e-6`，保证 odds 和梯度路径有限。

In [5]:
boundary_logp = torch.tensor(0.0)  # 构造概率恰好为一的数值边界。
naive_log_odds = boundary_logp - torch.log1p(-torch.exp(boundary_logp))  # 朴素公式在 log(0) 处产生正无穷。
safe_probability = torch.exp(boundary_logp).clamp(max=1.0 - 1.0e-6)  # 对概率上界施加与训练相同的保护。
safe_log_odds = boundary_logp - torch.log1p(-safe_probability)  # 用裁剪概率计算有限 log-odds。
naive_is_finite = bool(torch.isfinite(naive_log_odds).item())  # 检查错误实现的非有限结果。
safe_is_finite = bool(torch.isfinite(safe_log_odds).item())  # 检查修正实现的有限结果。
print(f"错误行为：logp=0时 naive_log_odds={naive_log_odds.item()}，finite={naive_is_finite}")  # 展示 odds 数值溢出。
print(f"修正行为：clamped_probability={safe_probability.item():.6f}，safe_log_odds={safe_log_odds.item():.5f}，finite={safe_is_finite}")  # 展示上界保护后的有限值。

错误行为：logp=0时 naive_log_odds=inf，finite=False
修正行为：clamped_probability=0.999999，safe_log_odds=13.80232，finite=True


## 6. 生产边界

条件 unigram 不是自回归 LLM。生产 ORPO 需要 prompt/response mask、逐 Token causal forward、chosen/rejected 长度策略、beta 搜索、验证集、防 reward hacking、混合精度稳定性、梯度累积、分布式 checkpoint 和独立安全评测。

In [6]:
training_diagnostics = {"pairs": len(preference_pairs), "vocab_size": len(vocabulary), "sft_final_loss": sft_history[-1]["loss"], "orpo_final_loss": orpo_history[-1]["loss"], "sft_mean_margin": sft_mean_margin, "orpo_mean_margin": orpo_mean_margin, "orpo_pair_accuracy": orpo_pair_accuracy, "reference_model_forwards": 0}  # 汇总训练质量和 ORPO 单模型属性。
print("生产监控快照：", training_diagnostics)  # 输出真实 ORPO 训练应监控的指标集合。

生产监控快照： {'pairs': 6, 'vocab_size': 66, 'sft_final_loss': 2.937997579574585, 'orpo_final_loss': 2.791879653930664, 'sft_mean_margin': 1.488532582918803, 'orpo_mean_margin': 2.3689893086751304, 'orpo_pair_accuracy': 1.0, 'reference_model_forwards': 0}


## 7. 最小回归测试

断言覆盖样本规模、真实梯度训练、同初始化、公平 margin 比较和数值保护。

In [7]:
assert len(preference_pairs) >= 5 and len(vocabulary) > 20  # 保证案例包含足够可读偏好对和非平凡词表。
assert sft_history[-1]["loss"] < sft_history[0]["loss"] and orpo_history[-1]["loss"] < orpo_history[0]["loss"]  # 保证两条路径都实际通过 forward/backward 学习。
assert all(row["gradient_norm"] > 0.0 and math.isfinite(row["gradient_norm"]) for row in orpo_history)  # 保证 ORPO 训练具有真实有限梯度。
assert orpo_mean_margin > sft_mean_margin  # 保证同一初始化和数据上显式偏好项产生更大平均 margin。
assert orpo_pair_accuracy >= sft_pair_accuracy and orpo_pair_accuracy >= 5.0 / 6.0  # 保证 margin 改善没有牺牲逐对排序。
assert not naive_is_finite and safe_is_finite  # 保证 odds 溢出失败真实复现并被数值保护修正。
assert training_diagnostics["reference_model_forwards"] == 0  # 保证实现没有暗中调用第二个 Reference Model。